In [1]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from dotenv.ipython import load_dotenv

In [71]:
load_dotenv(override=True)

True

In [9]:
llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0)

In [10]:
agent = create_agent(
model=llm,
system_prompt="You are a helpful assistant"
)

In [ ]:
resp = agent.invoke(input={"messages":[
    {"role":"user", "content":"Je m'appel Taoufik KEHAL"}
]})

In [12]:
print(resp['messages'][-1].content)

Bonjour Taoufik ! Enchanté de faire votre connaissance. Comment puis-je vous aider aujourd'hui ?


In [13]:
resp = agent.invoke(input={"messages":[
    {"role":"user", "content":"Comment je m'appel"}
]})

In [14]:
print(resp['messages'][-1].content)

Je suis ravi de vous rencontrer ! Je suis un assistant virtuel, donc je n'ai pas de nom personnel, mais je suis là pour vous aider et répondre à vos questions. Qu'est-ce que vous voulez savoir ou discuter ?


In [24]:
from langchain.agents.middleware import ModelRequest,ModelResponse,wrap_model_call
from langchain.messages import HumanMessage, SystemMessage, AIMessage

In [26]:
basic_llm =ChatGroq(model="meta-llama/llama-prompt-guard-2-86m",temperature=0)
advanced_llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0)

In [31]:
@wrap_model_call
def dynamic_model_selection(request:ModelRequest, hundler) -> ModelResponse:

    env = request.runtime.context.get("env", "test")
    if env == "prod":
        model =advanced_llm
        print("advanced llm selected")
    else:
    
        model = basic_llm
        print("basic llm selected")

    return hundler(request.override(model=model))

In [40]:
agent2 = create_agent(
model=basic_llm,
tools=[],
middleware=[dynamic_model_selection],
debug =True
)

In [41]:
resp = agent2.invoke(
    input={"messages":[HumanMessage("c'est auoi un agent AI")]},
    context={"env":"prod"})

[values] {'messages': [HumanMessage(content="c'est auoi un agent AI", additional_kwargs={}, response_metadata={}, id='bee16eda-f0bd-48e7-8798-b82d302e6f25')]}
advanced llm selected
[updates] {'model': {'messages': [AIMessage(content="Bonjour ! Je suis un agent AI, également appelé chatbot ou modèle de langage artificiel. Je suis conçu pour comprendre et répondre à vos questions, discuter avec vous sur divers sujets, et même vous aider à résoudre des problèmes.\n\nJe suis basé sur des algorithmes de machine learning qui me permettent d'apprendre et d'améliorer mes capacités de compréhension et de réponse au fil du temps. Je peux comprendre et générer du texte en français, ainsi que d'autres langues.\n\nJe suis là pour vous aider, répondre à vos questions, et discuter avec vous sur des sujets tels que :\n\n- La science et la technologie\n- L'histoire et la culture\n- La littérature et l'art\n- La musique et le cinéma\n- Les actualités et les événements du monde\n- Et bien d'autres choses

In [ ]:
from IPython.display import Markdown

In [43]:
Markdown(resp["messages"][-1].content)

Bonjour ! Je suis un agent AI, également appelé chatbot ou modèle de langage artificiel. Je suis conçu pour comprendre et répondre à vos questions, discuter avec vous sur divers sujets, et même vous aider à résoudre des problèmes.

Je suis basé sur des algorithmes de machine learning qui me permettent d'apprendre et d'améliorer mes capacités de compréhension et de réponse au fil du temps. Je peux comprendre et générer du texte en français, ainsi que d'autres langues.

Je suis là pour vous aider, répondre à vos questions, et discuter avec vous sur des sujets tels que :

- La science et la technologie
- L'histoire et la culture
- La littérature et l'art
- La musique et le cinéma
- Les actualités et les événements du monde
- Et bien d'autres choses encore !

Qu'est-ce que vous voulez discuter aujourd'hui ?

In [ ]:

agent = create_agent(
model=llm,
system_prompt="You are a helpful assistant"

)

In [47]:
resp =agent.invoke(input={"messages":[HumanMessage("Je m'appelle Taoufik")]})



In [48]:
print(resp['messages'][-1].content)

Enchanté, Taoufik ! Je suis ravi de faire votre connaissance. Comment puis-je vous aider aujourd'hui ?


In [49]:
resp =agent.invoke(input={"messages":[HumanMessage("C'est quoi mon nom")]})


In [50]:
print(resp['messages'][-1].content)

Je suis désolé, mais je ne connais pas votre nom. Nous venons de commencer notre conversation, et je n'ai aucune information sur vous. Si vous voulez partager votre nom avec moi, je serais ravi de le connaître !


In [54]:
from langgraph.checkpoint.memory import InMemorySaver

In [55]:
memory = InMemorySaver()
agent = create_agent(
model=llm,
system_prompt="You are a helpful assistant",
checkpointer= memory
)

In [58]:
config= {"configurable":{"thread_id":1}}
resp =agent.invoke(input={"messages":[HumanMessage("Je m'appelle Taoufik")]},
                   config = config)


In [59]:
print(resp['messages'][-1].content)

Enchanté, Taoufik ! Je suis ravi de faire votre connaissance. Comment puis-je vous aider aujourd'hui ?


In [60]:
config= {"configurable":{"thread_id":1}}
resp =agent.invoke(input={"messages":[HumanMessage("Comment je m''appelle")]},
                   config = config)

In [61]:
print(resp['messages'][-1].content)

Vous vous appelez Taoufik ! C'est un beau prénom arabe, n'est-ce pas ? Qu'est-ce que vous aimez faire, Taoufik ?


In [62]:
from langchain.tools import tool

In [63]:
@tool
def get_weather(city: str):
    """
    Get the wather of the given city
    """
    print("weather Tool invoked")
    return {"city":city,
            "temperature":23,
            "humidity":80}

In [64]:
@tool
def get_employee_info(employe_name: str):
    """
    Get infos about the given employee (salary, seniority)
    """
    print("get_employee_info Tool invoked")
    return {"name":employe_name,
            "salary":340000,
            "seniority":8}

In [65]:
agent4 = create_agent(
    model=llm,
    tools=[get_weather, get_employee_info],
    checkpointer=memory,
    system_prompt="answer user qustion using provided tools"
)

In [67]:
resp = agent4.invoke(input={"messages":HumanMessage("La meteo a casablanca")}, config = config)

weather Tool invoked


In [68]:
print(resp['messages'][-1].content)

La météo à Casablanca est chaude et humide avec une température de 23 degrés et une humidité de 80%.


In [69]:
resp = agent4.invoke(input={"messages":HumanMessage("Les informations de l'empoyee Taoufik")}, config = config)

get_employee_info Tool invoked


In [70]:
print(resp['messages'][-1].content)

Les informations de l'employé Taoufik sont les suivantes : nom : Taoufik, salaire : 340 000, ancienneté : 8 ans.


In [72]:
from langchain_tavily import TavilySearch

In [75]:
tavily =TavilySearch(maxresults =10, search_depth="advanced")

In [90]:
@tool
def web_search(query:str, maxresult =10):
    """
    Searche fo general current web results
    """
    print("search web invoked")
    results = tavily.invoke({"query":query})
    
    return results

In [91]:
agent5 = create_agent(
    model=llm,
    tools=[get_weather, get_employee_info,web_search],
    system_prompt="answer user qustion using provided tools"
)

In [96]:
resp =agent5.invoke(input={"messages":HumanMessage("Actualites sur le maroc")},)

search web invoked


In [97]:
print(resp["messages"][-1].content)

Il semble que le Maroc est actuellement en train de vivre des moments importants en termes d'économie, de politique et de société. Voici quelques-unes des informations qui ressortent de l'actualité :

* Le Maroc a été désigné pour accueillir le 77ème Congrès électif de la FIFA en 2027.
* Le gouvernement marocain a mis en avant son bilan social pour la Fête du Travail.
* Le Consulat général des États-Unis a inauguré son nouveau complexe à Casablanca Finance City.
* Le Maroc est dans le Top 4 de la région Afrique & Moyen-Orient en termes d'éolien.
* Les prix des carburants ont baissé à partir du 1er mai, avec le gasoil et l'essence repassant sous les 15 DH/L.

Il est important de noter que ces informations sont basées sur les résultats de la recherche et peuvent ne pas être exhaustives ou à jour. Il est toujours recommandé de vérifier les informations auprès de sources fiables pour obtenir une vision plus complète de la situation.
